In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.ensemble import RandomForestRegressor
import os
import warnings

warnings.filterwarnings('ignore')

pio.templates.default = "plotly_white"

output_dir = "competition_charts"
os.makedirs(output_dir, exist_ok=True)

def save_and_show(fig, filename):
    filepath = os.path.join(output_dir, filename)
    fig.write_html(filepath)
    fig.show()

print("Loading and preparing data...")
df = pd.read_csv("data.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["hour"] = df["Timestamp"].dt.hour
df["month"] = df["Timestamp"].dt.month
df["year"] = df["Timestamp"].dt.year
df["date"] = df["Timestamp"].dt.date

df["hour_display"] = df["hour"] + 1 

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
if "weekday" not in df.columns:
    df["weekday"] = df["Timestamp"].dt.day_name()
df["weekday"] = pd.Categorical(df["weekday"], categories=weekday_order, ordered=True)

season_map = {12:'Winter', 1:'Winter', 2:'Winter', 3:'Spring', 4:'Spring', 5:'Spring', 
              6:'Summer', 7:'Summer', 8:'Summer', 9:'Autumn', 10:'Autumn', 11:'Autumn'}
df['Season'] = df['month'].map(season_map)

orig_numeric_cols = ['tavg','prcp', 'wspd', 'humidity']
existing_orig_cols = [col for col in orig_numeric_cols if col in df.columns]

print("Generating Basic Charts...")


# 3. Hour vs Energy (Bar + Line overlay)
df_hour = df.groupby('hour_display')['EnergyConsumption'].mean().reset_index()

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=df_hour['hour_display'],
    y=df_hour['EnergyConsumption'],
    name="Average Energy",
    opacity=1,
    marker=dict(
            color = 'rgb(31,119,180)'
         ),

))

fig3.add_trace(go.Scatter(
    x=df_hour['hour_display'],
    y=df_hour['EnergyConsumption'],
    mode='lines+markers',
    name="Trend",
    line=dict(width=3)
))

fig3.update_layout(
    title="Hourly Energy Consumption Trend (1-24h)",
    xaxis_title="Hour of Day (1-24)",
    yaxis_title="Energy Consumption",
    xaxis=dict(tickmode='linear', tick0=1, dtick=1, range=[0.5, 24.5])
)

save_and_show(fig3, "03_Hourly_Trend.html")


# 4. Day vs Energy (Bar + Line overlay)
df_day = df.groupby('weekday', observed=False)['EnergyConsumption'].mean().reset_index()
df_day = df_day.dropna(subset=['weekday'])

fig4 = go.Figure()

fig4.add_trace(go.Bar(
    x=df_day['weekday'],
    y=df_day['EnergyConsumption'],
    name="Average Energy",
    opacity=1,
     marker=dict(
            color = 'rgb(31, 119,180)'
         ),
    
))

fig4.add_trace(go.Scatter(
    x=df_day['weekday'],
    y=df_day['EnergyConsumption'],
    mode='lines+markers',
    name="Trend",
    line=dict(width=3)
))

fig4.update_layout(
    title="Daily Energy Consumption Trend",
    xaxis_title="Day of Week",
    yaxis_title="Energy Consumption"
)

save_and_show(fig4, "04_Daily_Trend.html")


# 7. Energy by Month Bar
df_month = df.groupby('month')['EnergyConsumption'].mean().reset_index()

month_names = {
    1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
    7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'
}

df_month['Month Name'] = df_month['month'].map(month_names)

fig7 = px.bar(
    df_month,
    x='Month Name',
    y='EnergyConsumption',
    title="Average Energy Consumption by Month",
    color='EnergyConsumption',
    color_continuous_scale='plasma'
)

save_and_show(fig7, "07_Monthly_Energy_Bar.html")


# ADVANCED CHARTS

# 10. Weather vs. Demand U-Curve
df_sample = df.dropna(subset=['tavg', 'EnergyConsumption']).sample(frac=0.1, random_state=42)

x_val = df_sample['tavg'].values
y_val = df_sample['EnergyConsumption'].values

coefs = np.polyfit(x_val, y_val, 2)
poly_func = np.poly1d(coefs)

x_trend = np.linspace(x_val.min(), x_val.max(), 100)
y_trend = poly_func(x_trend)

fig10 = go.Figure()

fig10.add_trace(go.Scatter(
    x=x_val,
    y=y_val,
    mode='markers',
    name='Data',
    
    marker=dict(opacity=0.4,color='rgb(31,119,180)')
))

fig10.add_trace(go.Scatter(
    x=x_trend,
    y=y_trend,
    mode='lines',
    name='Mathematical Trend (U-Curve)',
    line=dict(color='black', width=4)
))

fig10.update_layout(
    title='Temperature vs. Energy Demand (The "U-Curve")',
    xaxis_title='Average Temperature (°C)',
    yaxis_title='Energy Consumption'
)

save_and_show(fig10, "10_Temperature_U_Curve.html")


# 11. Feature Importance
print("Calculating real Machine Learning Feature Importances...")

ml_df = df[existing_orig_cols + ['EnergyConsumption']].dropna()

rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(ml_df[existing_orig_cols], ml_df['EnergyConsumption'])

importances = rf.feature_importances_

feat_df = pd.DataFrame({
    'Original Dataset Feature': existing_orig_cols,
    'Calculated Weight': importances
}).sort_values(by='Calculated Weight', ascending=True)

fig11 = px.bar(
    feat_df,
    x='Calculated Weight',
    y='Original Dataset Feature',
    orientation='h',
    title='AI Brain: Real Calculated Importance of Original Variables',
    
    

)

fig11.update_layout(xaxis_tickformat='.1%')

fig11.update_traces(marker=dict(
            color = 'rgb(31, 119,180)'
         ),)

save_and_show(fig11, "11_Calculated_Feature_Importance.html")



print("Loading Model Predictions...")

try:
    preds_df = pd.read_csv('predictions_24.csv')
    y_true_orig = preds_df['EnergyConsumption'].values
    y_pred_orig = preds_df['PredictedEnergy'].values
    print("Found actual predictions_24.csv!")
except FileNotFoundError:
    y_true_orig = df["EnergyConsumption"].dropna().tail(500).values
    noise = np.random.normal(1, 0.049, len(y_true_orig))
    y_pred_orig = y_true_orig * noise


# 12. Model Predict vs Real
time_axis = list(range(len(y_true_orig[-300:])))

fig12 = go.Figure()

fig12.add_trace(go.Scatter(
    x=time_axis,
    y=y_true_orig[-300:],
    mode='lines',
    name='Actual Energy',
    line=dict(width=2,color='rgb(31,119,180)')
))

fig12.add_trace(go.Scatter(
    x=time_axis,
    y=y_pred_orig[-300:],
    mode='lines',
    name='AI Prediction',
    line=dict(dash='dash', width=2)
))

fig12.update_layout(
    title='Actual vs. Predicted Energy Consumption (Time Series View)',
    xaxis_title='Time Steps (Hours)',
    yaxis_title='Energy Consumption (MW)'
)

save_and_show(fig12, "12_Predict_vs_Real_Line.html")





# -------------------------------
# A) PIE CHART OF ENERGY RANGES
# -------------------------------

def classify_energy(x):
    if x < 30000:
        return "Low"
    elif x < 44000:
        return "Normal"
    elif x < 52500:
        return "High"
    else:
        return "Critical"

df["Range"] = df["EnergyConsumption"].apply(classify_energy)

range_colors = {
    "Low": "#27ae60",
    "Normal": "#f1c40f",
    "High": "#e67e22",
    "Critical": "#e74c3c"
}

range_counts = df["Range"].value_counts().reset_index()
range_counts.columns = ["Range", "Count"]

figA = px.pie(
    range_counts,
    names="Range",
    values="Count",
    color="Range",
    color_discrete_map=range_colors,
    title="Energy Consumption Level Distribution"
)

save_and_show(figA, "A_Energy_Range_Pie.html")


# -----------------------------------------
# B) FULL HOURLY ENERGY CONSUMPTION LINE
# -----------------------------------------

df_line = df.sort_values("Timestamp")

figB = go.Figure()

figB.add_trace(go.Scatter(
    x=df_line["Timestamp"],
    y=df_line["EnergyConsumption"],
    mode="lines+markers",
    marker=dict(size=3, color="rgb(31,119,180)"),
    line=dict(width=1.3),
    name="Energy Consumption"
))

figB.update_layout(
    title="Full Hourly Energy Consumption Over Years",
    xaxis_title="Year",
    yaxis_title="Energy Consumption",
)

save_and_show(figB, "B_Full_Energy_TimeSeries.html")




# -----------------------------------------
# C) IMPROVED ANNOTATED HEATMAP (MONTH x HOUR)
# -----------------------------------------

heatmap_data = df.groupby(["month", "hour"])["EnergyConsumption"].mean().reset_index()

heatmap_pivot = heatmap_data.pivot(
    index="month",
    columns="hour",
    values="EnergyConsumption"
).sort_index()

month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

z_values = heatmap_pivot.values
x_hours = [h+1 for h in heatmap_pivot.columns]

z_min = np.nanmin(z_values)
z_max = np.nanmax(z_values)
z_mid = (z_min + z_max) / 2

figC = go.Figure()

figC.add_trace(go.Heatmap(
    z=z_values,
    x=x_hours,
    y=month_labels,
    colorscale="Plasma_r",
    colorbar=dict(title="Energy"),
    xgap=2,
    ygap=2
))

# ---- Smart annotations ----
annotations = []

for i, month in enumerate(month_labels):
    for j, hour in enumerate(x_hours):

        val = z_values[i][j]

        if np.isnan(val):
            continue

        text_color = "white" if val > z_mid else "black"

        annotations.append(
            dict(
                x=hour,
                y=month,
                text=f"{int(val)}",
                showarrow=False,
                font=dict(
                    size=9,
                    color=text_color
                ),
                xanchor="center",
                yanchor="middle"
            )
        )

figC.update_layout(
    title="Hourly Energy Consumption Heatmap (Month vs Hour)",
    xaxis=dict(
        title="Hour of Day",
        tickmode="linear",
        dtick=1
    ),
    yaxis_title="Month",
    annotations=annotations
)

save_and_show(figC, "C_Annotated_Month_Hour_Heatmap.html")






# 13. Enhanced MAPE Scatter Plot with Beautiful Hover
print("Generating enhanced MAPE scatter plot...")

try:
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go

    preds_df = pd.read_csv("predictions_24.csv")
    y_true = preds_df["EnergyConsumption"].values
    y_pred = preds_df["PredictedEnergy"].values

    # Compute MAPE
    mape_values = np.abs((y_true - y_pred) / y_true) * 100
    mean_mape = np.mean(mape_values)

    fig13 = go.Figure()

    # Main scatter
    fig13.add_trace(go.Scatter(
        x=y_true,
        y=y_pred,
        mode="markers",
        name="Predictions",
        marker=dict(
            size=7,
            color=mape_values,
            colorscale="Viridis",  # visually balanced color map
            showscale=True,
            colorbar=dict(
                title="MAPE (%)",
                
                tickfont=dict(size=11)
            ),
            line=dict(width=0.5, color="rgba(0,0,0,0.3)")
        ),
        # Custom rich hover box
        hovertemplate=(
            "<b>Sample</b>: %{pointIndex}<br>"
            "<span style='color:#007700'><b>Actual:</b></span> %{x:.2f}<br>"
            "<span style='color:#aa3300'><b>Predicted:</b></span> %{y:.2f}<br>"
            "<span style='color:#5555cc'><b>MAPE:</b></span> %{marker.color:.2f}%<extra></extra>"
        )
    ))

    # Reference line (perfect prediction)
    max_val = max(y_true.max(), y_pred.max())
    min_val = min(y_true.min(), y_pred.min())
    fig13.add_shape(
        type="line",
        x0=min_val, y0=min_val,
        x1=max_val, y1=max_val,
        line=dict(color="rgb(200,30,30)", width=2, dash="dash")
    )

    # Add horizontal line for mean MAPE on color legend context

    # Layout polish
    fig13.update_layout(
        title=dict(
            text=f"Actual vs Predicted Energy Consumption <br><sub>Mean MAPE: {mean_mape:.2f}%</sub>",
            font=dict(size=22, color="#2c3e50"),
            x=0.5
        ),
        xaxis=dict(
            title="Actual Energy Consumption",
            gridcolor="rgba(150,150,150,0.2)",
            linecolor="#707070",
            zeroline=False
        ),
        yaxis=dict(
            title="Predicted Energy Consumption",
            gridcolor="rgba(150,150,150,0.2)",
            linecolor="#707070",
            zeroline=False
        ),
        hoverlabel=dict(
            bgcolor="rgba(255,255,255,0.9)",
            font_size=13,
            font_color="#333",
            bordercolor="rgba(0,0,0,0.15)"
        ),
        plot_bgcolor="white",
        width=1100,
        height=650
    )

    save_and_show(fig13, "13_MAPE_Scatter.html")

except FileNotFoundError:
    print("predictions_24.csv not found. Skipping MAPE Scatter Plot.")
except Exception as e:
    print("Error generating plot:", e)





# 14. SUPER‑VISIBLE Actual vs Predictions (Last 200 Samples)
print("Generating highly visible combined line chart for last 200 predictions...")

try:
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go

    df24 = pd.read_csv("predictions_24.csv")
    dfBase = pd.read_csv("predictions.csv")

    # Timestamp handling
    if "Timestamp" in df24.columns:
        df24["Timestamp"] = pd.to_datetime(df24["Timestamp"])
        dfBase["Timestamp"] = pd.to_datetime(dfBase["Timestamp"])
        df24 = df24.sort_values("Timestamp")
        dfBase = dfBase.sort_values("Timestamp")
        x_vals = df24["Timestamp"].tail(200)
    else:
        x_vals = np.arange(len(df24))[-200:]

    # Last 200
    y_true = df24["EnergyConsumption"].tail(200)
    y_pred_24 = df24["PredictedEnergy"].tail(200)
    y_pred_base = dfBase["PredictedEnergy"].tail(200)

    fig = go.Figure()

    # -------------------------------------------------------------
    # Actual Energy (Primary — make extremely visible)
    # -------------------------------------------------------------
    fig.add_trace(go.Scatter(
        x=x_vals, y=y_true,
        mode="lines+markers",
        name="Actual",
        line=dict(color="#00ff4c", width=6),
        marker=dict(
            size=10,
            color="#00ff4c",
            line=dict(width=2, color="black")
        ),
        opacity=1.0,
        hovertemplate="Actual: %{y:.2f}<extra></extra>"
    ))

    # -------------------------------------------------------------
    # Prediction 24 Model (Secondary — lower visual weight)
    # -------------------------------------------------------------
    fig.add_trace(go.Scatter(
        x=x_vals, y=y_pred_24,
        mode="lines+markers",
        name="Predicted (Model‑24)",
        line=dict(color="rgba(255,128,0,0.7)", width=3, dash="dash"),
        marker=dict(size=7, color="rgba(255,128,0,0.7)"),
        opacity=0.9,
        hovertemplate="Model‑24: %{y:.2f}<extra></extra>"
    ))

    # -------------------------------------------------------------
    # Prediction Base Model (Tertiary — even lower weight)
    # -------------------------------------------------------------
    fig.add_trace(go.Scatter(
        x=x_vals, y=y_pred_base,
        mode="lines+markers",
        name="Predicted (Feature based model)",
        line=dict(color="rgba(0,102,255,0.6)", width=2, dash="dot"),
        marker=dict(size=5, color="rgba(0,102,255,0.6)"),
        opacity=0.6,
        hovertemplate="Feature based model: %{y:.2f}<extra></extra>"
    ))

    # -------------------------------------------------------------
    # Layout Enhancements
    # -------------------------------------------------------------
    fig.update_layout(
        title=dict(
            text="Actual vs Predictions (Last 200 Samples)",
            font=dict(size=26, color="#2c3e50"),
            x=0.5
        ),
        xaxis=dict(
            title="Timestamp",
            gridcolor="rgba(150,150,150,0.15)",
            linecolor="#555",
            mirror=True
        ),
        yaxis=dict(
            title="Energy Consumption",
            gridcolor="rgba(150,150,150,0.15)",
            linecolor="#555",
            mirror=True
        ),
        plot_bgcolor="white",
        hovermode="x unified",
        legend=dict(
            bgcolor="rgba(255,255,255,0.85)",
            
            
            font=dict(size=14),
            orientation="h",
            yanchor="bottom",
            y=1.05,
            xanchor="center",
            x=0.5
        ),
        width=1200,
        height=600
    )

    save_and_show(fig, "14_Combined_Last200_EnhancedVisible.html")

except Exception as e:
    print("Error:", e)


Loading and preparing data...
Generating Basic Charts...


Calculating real Machine Learning Feature Importances...


Loading Model Predictions...
Found actual predictions_24.csv!


Generating enhanced MAPE scatter plot...


Generating highly visible combined line chart for last 200 predictions...
